# Sentiment Classification Project

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Verify Setup
Make sure cuda (GPU) is available

In [13]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 5060 Ti


# Load data

In [14]:
train_full = pd.read_csv("/cluster/courses/cil/text-classification/data/train.csv")

# Data preprocessing TODO
We need to preprocess data. Step that come to my mind:
 - Remove word count outliers. (The vibe of a review comes across after 100 words or so).
 - We have german and english data. Should we translate everything to english
 - Should we search for smiley an insert text for them.
 - Should we search for ** (bold markers) and emphasize this word differenetly? 

# Build Validation Set
Todo: 

Currently we use 90% of the reviews for training, and the remaining 10% for validation
This should be optimized. Use k-fold validation or something like that to get most of our data as training and enable proper hyperparameter tuning later on.

In [15]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Baselines
The current baseline implementation follows the structure:
$$\text{Output} = \text{Classifier}(\text{EmbeddingForTokens}(x))$$

In this specific case:
* **EmbeddingForTokens**: They use `CountVectorizer`, which creates a "Bag of Words" representation (counting word frequencies).
* **Classifier**: They use **Logistic Regression**.

We could implement additional baselines by choosing more complex **EmbeddingForTokens** methods (such as word-level embeddings or pre-trained vectors) and more sophisticated **Classifier** models (such as Random Forests or simple Neural Networks (MLP)).

# TRAIN PIPELINE

## Overview
A modular training pipeline that evaluates combinations of (vectorizer, classifier) pairs.
Defined across three files: embeddings.py, models.py, and train_loop.py.

## Step 1 — Vectorization (embeddings.py)
For classical vectorizers (BoW, TF-IDF), the signature is:
    (X_train, X_val) -> (X_train_embedded, X_val_embedded)
The vocabulary is fit on X_train only, then applied to both splits.

For pretrained embedding models (BERT, GloVe), sentences are encoded
independently with no fitting step:
    X -> X_embedded

## Step 2 — Classification (models.py)
Each classifier is a factory function that returns a fresh, untrained model instance.
Keeping them as functions (rather than instances) ensures each combination in the
loop starts from a clean state.

model is then used to fit X_train on Y_train: model.fit(X_train, Y_train)

## Step 3 — Train Loop (train_loop.py)
Takes a list of (vectorizer_fn, model_fn) tuples. For each combination it:
  1. Vectorizes the data
  2. Trains the classifier on the training embeddings X_train, Y_train
  3. Evaluates on both train and validation splits

Returns a list of result dicts, one per combination:
    {
        "vectorizer":          str,   # name of the vectorizer function
        "model":               str,   # name of the model factory
        "training_score":      float,
        "training_mae":        float,
        "training_accuracy":   float,
        "validation_score":    float,
        "validation_mae":      float,
        "validation_accuracy": float
    }

In [16]:
# # so that it reloads the modules every time we run this cell, so we always have the latest version of the code
# %load_ext autoreload
# %autoreload 2

# import importlib
# import train_loop
# importlib.reload(train_loop)
# from train_loop import train_loop

In [17]:
# NOTE: don't do GloVe, as only for English
# downlaod GloVe embeddings (100-dimensional vectors) and extract the file
# import urllib.request
# import zipfile

# url = "http://nlp.stanford.edu/data/glove.6B.zip"
# urllib.request.urlretrieve(url, "glove.6B.zip")
# with zipfile.ZipFile("glove.6B.zip", "r") as f:
#     f.extract("glove.6B.100d.txt")  # 100-dimensional vectors

## Simple baselines

As baselines, we first evaluate these combinations and then try to beat it later with more sophisticated approaches.
1) BoW  +  Logistic Regression
2) TF-IDF  +  Logistic Regression
3) multilingual pre-trained embeddings + Logistic Regression

Notes: MLP and Random Forest take way too long on such sparse and high dimensional embeddings as BoW and TF-IDF... Skipped for now

In [18]:
# import sys
# import subprocess
# subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers"])

In [19]:
!find /home/taekim -name "embeddings.py" 2>/dev/null
!cat embeddings.py

/bin/bash: line 1: cat: No such file or directory


In [20]:
import os
os.chdir('/home/taekim/Garnella')

In [21]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_capability())

2.11.0+cu126
True
(12, 0)


In [22]:
from embeddings import *
from models import *
from train_loop_caching import train_loop
from functools import partial

get_qwen_saved = partial(get_qwen_embeddings, save=True)

combinations = [
    # baselines
    (get_tfidf_embeddings, get_logistic_regression),
    (get_multilingual_embeddings, get_logistic_regression),
    # qwen
    (get_qwen_embeddings, get_logistic_regression),
    (get_qwen_embeddings, get_linear_svm),
    (get_qwen_embeddings, get_mlp),
    (get_qwen_embeddings, get_random_forest),
    (get_qwen_embeddings, get_xgboost),
    (get_qwen_embeddings, get_knn),
]

results = train_loop(train_df, val_df, combinations)
pd.DataFrame(results).sort_values("validation_score", ascending=False)

Done with embeddings: get_tfidf_embeddings
Combination: get_tfidf_embeddings + LogisticRegression
Training Score: 0.8841, MAE: 0.4636, Accuracy: 0.6354
Validation Score: 0.8666, MAE: 0.5334, Accuracy: 0.5772


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10696.88it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches:   1%|          | 8/886 [02:06<3:50:37, 15.76s/it]


KeyboardInterrupt: 